In [1]:
import tensorflow as tf
import numpy as np
import gym
from ou_noise import OUNoise
import random


LAYER_1 = 400
LAYER_2 = 300
LAYER_3 = 300
keep_rate = 0.8
LAMBDA = 0.00001 # regularization term
GAMMA = 0.99
class DDPG(object):


    def __init__(self, sess, state_dim, action_dim, max_action, min_action, actor_learning_rate, critic_learning_rate, tau, RANDOM_SEED, device = '/cpu:0'):

        self.sess = sess
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        self.s_dim = state_dim
        self.a_dim = action_dim
        self.actor_learning_rate = actor_learning_rate
        self.critic_learning_rate = critic_learning_rate
        self.tau = tau
        self.device = device
        self.max_action = max_action
        self.min_action = min_action
        # Placeholders
        self.inputs = tf.placeholder(tf.float32, shape=[None, self.s_dim], name='state')
        self.action = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='actions')
        scope = 'net'    
        self.v, self.a, self.scaled_a, self.saver = self._build_net(scope)
        self.a_params = tf.trainable_variables(scope=scope + '/actor')
        self.c_params = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')
        scope = 'target'    
        self.v_target, self.a_target, self.scaled_a_target, self.saver_target = self._build_net(scope)
        self.a_params_target = tf.trainable_variables(scope=scope + '/actor')
        self.c_params_target = tf.trainable_variables(scope=scope + '/critic')
        #self.a_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/actor')
        #self.c_params_target = tf.get_collection(tf.GraphKeys.TRAINABLE_VARIABLES, scope=scope + '/critic')

        
        with tf.variable_scope('learning_rate'): 
            # global step
            self.global_step = tf.Variable(0, trainable=False)
            self.actor_decay_learning_rate = tf.train.exponential_decay(self.actor_learning_rate, self.global_step, 100000, 0.96, staircase=True)
            self.critic_decay_learning_rate = tf.train.exponential_decay(self.critic_learning_rate, self.global_step, 100000, 0.96, staircase=True)
        
        with tf.device(self.device):
            # Op for periodically updating target network with online network
            # weights with regularization
            self.generate_param_updater()
           
            self.predicted_q_value = tf.placeholder(tf.float32, [None, 1])
            # Define loss and optimization Op
            self.squared = tf.square(tf.subtract(self.predicted_q_value,self.v))
            self.l2_loss = tf.losses.get_regularization_loss(scope="net/critic")
            self.loss = tf.reduce_mean(self.squared) + self.l2_loss 
            self.critic_optimize = tf.train.AdamOptimizer(self.critic_decay_learning_rate).minimize(self.loss, global_step=self.global_step) 
            self.action_grads = tf.gradients(self.v, self.action)[0]
            self.actor_gradients = tf.gradients(self.a, self.a_params, -self.action_grads)
            self.actor_optimize = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self.actor_gradients, self.a_params), global_step=self.global_step)
            
            # inverting gradients
            self.inverting_gradients_placeholder = tf.placeholder(tf.float32, shape=[None, self.a_dim], name='inverting_gradients')
            self._dq_da = tf.gradients(self.v, self.action)[0] # q, a 
            self._grad = tf.gradients(self.a, self.a_params, -self.inverting_gradients_placeholder)
            self._train_actor = tf.train.AdamOptimizer(self.actor_decay_learning_rate).apply_gradients(zip(self._grad, self.a_params),global_step=self.global_step)
            
            


    def _build_net(self,scope):
       
        with tf.device(self.device):        
            with tf.variable_scope(scope + '/critic'):
                
                
                '''
                net = tf.layers.dense(self.inputs, LAYER_1, tf.nn.relu, name='critic_L1')
                initializer = tf.variance_scaling_initializer()
                s_union_weights = tf.Variable(initializer.__call__([LAYER_1, LAYER_2]), name='critic_L2_Ws')
                a_union_weights = tf.Variable(initializer.__call__([self.a_dim, LAYER_2]), name='critic_L2_Wa')
                union_biases = tf.Variable(tf.zeros([LAYER_2]), name='critic_L2_b')
                net = tf.nn.relu(tf.matmul(net, s_union_weights) + tf.matmul(self.action, a_union_weights) + union_biases,name='critic_L2')
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                v = tf.layers.dense(net, self.a_dim, kernel_initializer=w_init, name='critic_output')
                '''
                regularizer = tf.contrib.layers.l2_regularizer(scale=LAMBDA)
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1, weights_regularizer=regularizer, activation_fn=tf.nn.leaky_relu)
                l2_a = tf.contrib.layers.fully_connected(self.action, LAYER_2, weights_regularizer=regularizer, activation_fn=None)
                l2_s = tf.contrib.layers.fully_connected(l1, LAYER_2, weights_regularizer=regularizer,activation_fn=None)
                l2 = tf.nn.leaky_relu(l2_s + l2_a)
                v = tf.contrib.layers.fully_connected(l2, 1, weights_regularizer=regularizer, activation_fn=None)
				
            with tf.variable_scope(scope + '/actor'):
                l1 = tf.contrib.layers.fully_connected(self.inputs, LAYER_1,  activation_fn=tf.nn.leaky_relu) # tf.nn.leaky_relu tf.nn.relu
                l2 = tf.contrib.layers.fully_connected(l1, LAYER_2,  activation_fn=tf.nn.leaky_relu)
                w_init = tf.random_uniform_initializer(minval=-0.003, maxval=0.003)
                a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=tf.nn.tanh) # (para el ddpg)
                #a = tf.contrib.layers.fully_connected(l2, self.a_dim, weights_initializer=w_init, activation_fn=None) # (para el inverted)
                scaled_a = tf.multiply(a,self.max_action) #(para el ddpg)
                #scaled_a = a # (para el inverted)
                
                       
        saver = tf.train.Saver()
        return v, a, scaled_a, saver

    def train(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # print(L2_LOSS)
        a_outs = self.predict_action(s_batch)
        self.actor_train(s_batch, a_outs)
        
        self.update_target_network()

        return

    def test_gradient(self, s_batch, a_batch, r_batch, t_batch, s2_batch, MINIBATCH_SIZE):
        
        
        # get q target
        target_q = self.critic_predict_target(s2_batch, self.predict_action_target(s2_batch))
        # obtain y
        y_i = []
        for k in range(MINIBATCH_SIZE):
            if t_batch[k]:
                y_i.append(r_batch[k])
            else:
                y_i.append(r_batch[k] + GAMMA * target_q[k])
        
        # train critic
        LOSS = self.critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        # train critic
        #ac_tor_grads = self._critic_train(s_batch, a_batch, np.reshape(y_i, (MINIBATCH_SIZE, 1)))
        #print('a grads',ac_tor_grads)
        actions = self.predict_action(s_batch)
        
        upper = self.max_action
        lower = self.min_action

        # get dq/da array, action array
        #print(upper, '***************')
        dq_das = self.sess.run([self._dq_da], feed_dict={self.inputs: s_batch, self.action:actions})[0]
        # inverting gradients, if dq_da >= 0, apply upper method, else lower method
        inverting_gradients = []
        #'''
        # print('1 dq_das, actions',dq_das, actions)
        '''
        # print('dq_das, actions',dq_das, actions)
        for dq_da, action in zip(dq_das, actions):
            # print('dq_da, action',dq_da, action)
            if dq_da >= 0.0:
                inverting_gradients.append(dq_da * (self.max_action - action) / (self.max_action - self.min_action))
            else:
                inverting_gradients.append(dq_da * (action - self.min_action) / (self.max_action - self.min_action))
        inverting_gradients = np.array(inverting_gradients).reshape(-1, 1)
		'''

        for i in range(MINIBATCH_SIZE):
            #print('2', i,dq_das[i])
            for j in range(self.a_dim):
                if dq_das[i][j] >= 0.0:
                    dq_das[i][j] = dq_das[i][j] * (self.max_action - actions[i][j]) / (self.max_action - self.min_action)
                else:
                    dq_das[i][j] = dq_das[i][j] * (actions[i][j] - self.min_action) / (self.max_action - self.min_action)
        
        # print(dq_das,inverting_gradients)
        # exit()
        inverting_gradients = dq_das 
        
        # print('2 dq_das, actions',dq_das, actions)
        
         
        #print('1','inverting_gradients',inverting_gradients)
        
        # print('2','inverting_gradients',inverting_gradients,dq_das, actions)
        # time.sleep(1)
        # update actor
        self.sess.run(self._train_actor, feed_dict={self.inputs: s_batch, self.inverting_gradients_placeholder: inverting_gradients})
        self.update_target_network()
        return

    
    def _critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.action_grads], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def update_target_network(self):
        self.sess.run([self.a_updater,self.c_updater])

    def generate_param_updater(self):
        self.a_updater = [self.a_params_target[i].assign(tf.multiply(self.a_params[i], self.tau) + tf.multiply(self.a_params_target[i], 1. - self.tau))
                for i in range(len(self.a_params))]
        self.c_updater = [self.c_params_target[i].assign(tf.multiply(self.c_params[i], self.tau) + tf.multiply(self.c_params_target[i], 1. - self.tau))
                for i in range(len(self.c_params))]

    def critic_train(self, inputs, action, predicted_q_value):
        return self.sess.run([self.loss,self.critic_optimize], feed_dict={
            self.inputs: inputs,
            self.action: action,
            self.predicted_q_value: predicted_q_value
        })


    def actor_train(self,inputs, action):
        return self.sess.run(self.actor_optimize, feed_dict={
            self.inputs: inputs,
            self.action: action
        })

    def save(self):
        self.saver.save(self.sess,"./model/model.ckpt")
        self.saver_target.save(self.sess,"./model/model_target.ckpt")
        print("Model saved in file: actor_model")

    
    def load(self):
        self.saver.restore(self.sess,"./model/model.ckpt")
        self.saver_target.restore(self.sess,"./model/model_target.ckpt")
        


    def critic_predict_target(self, state, action):
        return self.sess.run(self.v_target, feed_dict={
            self.inputs: state,
            self.action: action
        })
        
    def predict_action_target(self, state):
        return self.sess.run(self.scaled_a_target, feed_dict={
            self.inputs: state
        })

    def predict_action(self, state):
        return self.sess.run(self.scaled_a, feed_dict={
            self.inputs: state
        })



def normalizing_state(state):
    #state = [V,P] being V_min = 0, V_max = 210, P_min = 0, P_max = 25000 we use ((xi-x_min)/(x_max-x_min))-1
    V_min = 0
    V_max = 210
    P_min = 0
    P_max = 54000
    DeltaP_min = -15000
    DeltaP_max = 15000

    st = [(2*(state[0]-V_min)/(V_max-V_min))-1, (2*(state[1]-P_min)/(P_max-P_min))-1,(2*(state[1]-DeltaP_min)/(DeltaP_max-DeltaP_min))-1]

    return st


if __name__ == '__main__':
    from replay_buffer import ReplayBuffer
    import gym
    import gym_mppt
    ACTOR_LEARNING_RATE = 0.0001
    CRITIC_LEARNING_RATE =  0.001
    # Soft target update param
    TAU = 0.001
    DEVICE = '/cpu:0'
    # ENV_NAME = 'MountainCarContinuous-v0'
    ENV_NAME = 'mppt_shaded-v0'#'Pendulum-v0'
    # import gym_foo
    # ENV_NAME = 'nessie_end_to_end-v0'
    max_action = 10.
    min_action = -10.
#     epochs = 15000
    epochs = 20000
    epsilon = 1.0
    min_epsilon = 0.1
    EXPLORE = 15000
    BUFFER_SIZE = 50000
    RANDOM_SEED = 51234
    MINIBATCH_SIZE = 64# 32 # 5
    with tf.Session() as sess:
        np.random.seed(RANDOM_SEED)
        tf.set_random_seed(RANDOM_SEED)
        env = gym.make(ENV_NAME)
        state_dim = np.size(env.reset()) #env.observation_space.shape[0]
        action_dim = 1 #env.action_space.shape[0]
        ddpg = DDPG(sess, state_dim, action_dim, max_action, min_action, ACTOR_LEARNING_RATE, CRITIC_LEARNING_RATE, TAU, RANDOM_SEED,device=DEVICE)
        sess.run(tf.global_variables_initializer())
        #ddpg.load()
        replay_buffer = ReplayBuffer(BUFFER_SIZE, RANDOM_SEED)
        ruido = OUNoise(action_dim, mu = 0.0)
        Reward_episodios = []
        llegadas =0
        for i in range(epochs):
            state = env.reset()
            normalized_state = normalizing_state(state)
            #print('EL ESTADO RESETEADO ES', state, state.shape)
            #print('epoch = ', i)
            done = False
            epsilon -= (epsilon/EXPLORE)
            epsilon = np.maximum(min_epsilon,epsilon)
            episode_r = 0.
            step = 0 
            max_steps = 45 #es solo a efecto de contador
            r_episodio_actual = []
            #llegadas =0
            while (not done):
                #print('step =', step)
                #wait = input("PRESS ENTER TO CONTINUE.")
                action = ddpg.predict_action(np.reshape(normalized_state,(1,state_dim)))
                action1 = action
                #print('LA ACCION sin clipear ES', action1, action1.shape) 
                action = np.clip(action1,min_action,max_action)
                #action = action + max(epsilon,0)*ruido.noise()
                #e-greedy action selection
                if random.random() < epsilon:
                    action = random.uniform(min_action,max_action)
                    #print('Explorando')

                action = np.clip(action,min_action,max_action)
                #print('ruido =', max(epsilon,0)*ruido.noise(),'epsilon =',epsilon)
                #print('LA ACCION clipeada ES', action, action.shape)
                
                next_state, reward, done, info = env.step(action)
                normalized_next_state = normalizing_state(next_state)
                #print('EL NEXT_ESTADO ES', next_state, next_state.shape) 
                #reward = np.clip(reward,-1.,1.)
                #print('instaneous r = ',reward)
                #print('Steps = ', info['Steps'],'V0 =',info['v0'])
                #print('normalized_state =',normalized_state)
                #wait = input("PRESS ENTER TO CONTINUE.")
                replay_buffer.add(np.reshape(normalized_state, (state_dim,)), np.reshape(action, (action_dim,)), reward,
                                      done, np.reshape(normalized_next_state, (state_dim,)))
                state = next_state
                episode_r = episode_r + reward
                normalized_state = normalizing_state(state)

                r_episodio_actual.append(reward)
                if replay_buffer.size() > MINIBATCH_SIZE:
                    s_batch, a_batch, r_batch, t_batch, s2_batch = replay_buffer.sample_batch(MINIBATCH_SIZE)
                    # train ddpg normally:
                    ddpg.train(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
                    #train with inverted gradients
                    #ddpg.test_gradient(s_batch, a_batch, r_batch, t_batch, s2_batch,MINIBATCH_SIZE)
#                 print(i, step, 'last r', round(reward,3), 'episode reward',round(episode_r,3), 'epsilon', round(epsilon,3))
#                 print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state,'last r =', round(reward[0][0],3), 'episode reward =',round(episode_r[0][0],3), 'epsilon =', round(epsilon,3))
                
                step += 1
            
            
            #if info['Steps'] > max_steps:
             #   llegadas +=1
#             print ('--------------------------------------------')
#             print('epoch =',i,'step =' ,step, 'done =', done,'St(V,P,I) =',state, 'accion =',action,'last r =', reward, 'episode reward =',episode_r, 'epsilon =', round(epsilon,3))
#             print ('--------------------------------------------')

                

            Reward_episodios.append(r_episodio_actual)
            np.save('Reward_episodios_DDPG01.npy',Reward_episodios)
            

            print('epoch          = ', i)      
            print('Episode Reward = ', episode_r/3)
            print('Last Reward    = ', reward)
            print('State          = ', state)
            print()

            if i % 1000 == 0:      # Save model every 1000 iterations
                ddpg.save()
            
        ddpg.save()

C:\Users\archi\anaconda3\envs\BTP\lib\site-packages\gym\logger.py:30: UserWarning: WARN: Box bound precision lowered by casting to float32
  warnings.warn(colorize('%s: %s'%('WARN', msg % args), 'yellow'))


epoch          =  0
Episode Reward =  23.514626666666683
Last Reward    =  9e-05
State          =  [   7.4044185 2780.0706    2780.0706   ]

Model saved in file: actor_model
epoch          =  1
Episode Reward =  4.247630000000002
Last Reward    =  0.5759
State          =  [   93.281364 21959.31     -1390.8065  ]

epoch          =  2
Episode Reward =  35.39861999999999
Last Reward    =  0.04793
State          =  [  189.85744 15215.961   -1858.0735 ]

epoch          =  3
Episode Reward =  30.448240012170473
Last Reward    =  0.8084200024604797
State          =  [  168.83344 21251.178    1584.2793 ]

epoch          =  4
Episode Reward =  12.88563
Last Reward    =  0.06528
State          =  [  15.310417 5742.3335   -675.31744 ]

epoch          =  5
Episode Reward =  3.2353466666666644
Last Reward    =  0.06151
State          =  [   41.48672 15562.327   -1992.7031 ]

epoch          =  6
Episode Reward =  18.497543333333333
Last Reward    =  0.04241
State          =  [  151.97672 12598.614  

c:\users\archi\python code\btp\luis_upgrade\mppt-gym_v4\gym_mppt\envs\mppt_env_shaded0.py:37: RuntimeWarning: overflow encountered in exp
  I = self.Np * Iph - self.Np * Irs * (np.exp(self.q / (self.k * Tcell * self.A) * vx * (1/SH) / self.Ns) - 1)


epoch          =  42
Episode Reward =  11.765276676272649
Last Reward    =  0.0
State          =  [187.46935   0.        0.     ]

epoch          =  43
Episode Reward =  1.934913333333334
Last Reward    =  0.05434
State          =  [   69.31907 26000.078   -2780.0818 ]

epoch          =  44
Episode Reward =  5.657313333909918
Last Reward    =  0.15722
State          =  [   57.1956 13701.461  -1653.4545]

epoch          =  45
Episode Reward =  22.40287666666666
Last Reward    =  0.61245
State          =  [  117.958374 16551.85        87.53967 ]

epoch          =  46
Episode Reward =  3.8234066619500795
Last Reward    =  0.0077
State          =  [   24.888636  9336.581    -3601.2317  ]

epoch          =  47
Episode Reward =  13.635676666666662
Last Reward    =  0.1971
State          =  [   82.14341 13555.617    -633.6759 ]

epoch          =  48
Episode Reward =  41.756230015640256
Last Reward    =  0.8521000146865845
State          =  [  131.24489 20578.64     -317.41406]

epoch         

epoch          =  99
Episode Reward =  8.124186666666672
Last Reward    =  0.0017
State          =  [ 142.16537 3080.0005  1280.7607 ]

epoch          =  100
Episode Reward =  17.31640667349252
Last Reward    =  0.00349
State          =  [  12.168383 4553.2744   2235.6536  ]

epoch          =  101
Episode Reward =  17.492210000012008
Last Reward    =  0.09054
State          =  [   32.827637 12293.691     2683.8796  ]

epoch          =  102
Episode Reward =  5.9083733333333335
Last Reward    =  0.02631
State          =  [   28.389347 10653.568     3237.6174  ]

epoch          =  103
Episode Reward =  20.9015533661623
Last Reward    =  0.0
State          =  [209.20935   0.        0.     ]

epoch          =  104
Episode Reward =  18.277600050683557
Last Reward    =  0.3050000071525574
State          =  [  103.00311 11164.929    -336.16895]

epoch          =  105
Episode Reward =  18.033393390345058
Last Reward    =  0.0117
State          =  [   32.569088 12225.392     3089.1782  ]

epoch 

...

epoch          =  19760
Episode Reward =  87.65528995133116
Last Reward    =  0.7610800266265869
State          =  [   93.15902 17059.633   -1200.1035 ]

epoch          =  19761
Episode Reward =  20.31734996488861
Last Reward    =  0.9314799904823303
State          =  [   80.019356 22570.238     1678.9863  ]

epoch          =  19762
Episode Reward =  80.57997989939406
Last Reward    =  0.9500600099563599
State          =  [  120.586044 18491.764     -236.79883 ]

epoch          =  19763
Episode Reward =  79.6882932264125
Last Reward    =  0.9988700151443481
State          =  [   64.40079  13883.814       34.339844]

epoch          =  19764
Episode Reward =  72.65379349165596
Last Reward    =  0.8373600244522095
State          =  [7.2564598e+01 2.3841293e+04 1.8679688e+01]

epoch          =  19765
Episode Reward =  30.430123262107372
Last Reward    =  0.35093000531196594
State          =  [   56.235573 10190.199     2803.5781  ]

epoch          =  19766
Episode Reward =  88.977096663380

epoch          =  19813
Episode Reward =  91.49837334828528
Last Reward    =  0.8921499848365784
State          =  [   78.08248 27792.953    -731.86914]

epoch          =  19814
Episode Reward =  88.0550500036746
Last Reward    =  0.9934700131416321
State          =  [ 8.6873047e+01  2.9759076e+04 -2.4863281e+00]

epoch          =  19815
Episode Reward =  69.55227661669778
Last Reward    =  0.8064900040626526
State          =  [ 8.7145615e+01  2.9868213e+04 -2.3019531e+01]

epoch          =  19816
Episode Reward =  87.51687983159597
Last Reward    =  0.8437899947166443
State          =  [  135.21242 19127.273    -333.77734]

epoch          =  19817
Episode Reward =  67.9134397975613
Last Reward    =  0.30702999234199524
State          =  [   75.401535 17976.379    -1887.1367  ]

epoch          =  19818
Episode Reward =  48.93505341156075
Last Reward    =  0.37400001287460327
State          =  [   62.922123 13828.351    -2228.249   ]

epoch          =  19819
Episode Reward =  85.8405800

epoch          =  19866
Episode Reward =  80.27159009084183
Last Reward    =  0.9810600280761719
State          =  [  131.47298 18640.77      -76.48047]

epoch          =  19867
Episode Reward =  70.24746329581346
Last Reward    =  0.872409999370575
State          =  [1.1668065e+02 3.3282891e+04 1.7054688e+01]

epoch          =  19868
Episode Reward =  94.99222993087945
Last Reward    =  0.9969099760055542
State          =  [  119.70831 40618.605     723.84375]

epoch          =  19869
Episode Reward =  83.74298346085318
Last Reward    =  0.872219979763031
State          =  [  125.40777  18703.688      -22.195312]

epoch          =  19870
Episode Reward =  93.43586335594107
Last Reward    =  0.4972499907016754
State          =  [  111.41961 24958.674     171.96094]

epoch          =  19871
Episode Reward =  82.75943336014154
Last Reward    =  0.976859986782074
State          =  [  102.70343 22735.953     -70.6543 ]

epoch          =  19872
Episode Reward =  80.58701986606077
Last Rewar

epoch          =  19919
Episode Reward =  86.73851322875514
Last Reward    =  0.972100019454956
State          =  [  132.83858 29348.273      98.34961]

epoch          =  19920
Episode Reward =  87.93501997577066
Last Reward    =  1.0
State          =  [  128.37167 28153.285     107.36328]

epoch          =  19921
Episode Reward =  77.46531000391649
Last Reward    =  0.9986500144004822
State          =  [1.4058246e+02 2.1233500e+04 1.7558594e+00]

epoch          =  19922
Episode Reward =  67.4161800480603
Last Reward    =  0.43303999304771423
State          =  [   65.18957  14409.684      -23.714844]

epoch          =  19923
Episode Reward =  80.09314670524327
Last Reward    =  0.9743000268936157
State          =  [   97.87717 27125.816    -110.19922]

epoch          =  19924
Episode Reward =  76.42666667226392
Last Reward    =  0.9363999962806702
State          =  [   91.15712 20441.125     804.27344]

epoch          =  19925
Episode Reward =  35.9860399554988
Last Reward    =  0.2508

epoch          =  19972
Episode Reward =  77.74608325454756
Last Reward    =  0.9574999809265137
State          =  [  148.60214 30828.623    -310.68164]

epoch          =  19973
Episode Reward =  83.19193650964007
Last Reward    =  0.8129500150680542
State          =  [  104.19566 30050.887   -1210.6855 ]

epoch          =  19974
Episode Reward =  83.2440599873856
Last Reward    =  0.9440600275993347
State          =  [  154.81015 22091.459    -222.9336 ]

epoch          =  19975
Episode Reward =  81.66396654983691
Last Reward    =  0.946120023727417
State          =  [  135.51277 37905.156     963.1133 ]

epoch          =  19976
Episode Reward =  85.83523657762755
Last Reward    =  0.8090500235557556
State          =  [   59.613976 13745.706     -687.2334  ]

epoch          =  19977
Episode Reward =  85.18234987194107
Last Reward    =  0.9992799758911133
State          =  [  117.28447 24918.621     271.61328]

epoch          =  19978
Episode Reward =  0.1759033335656568
Last Reward   

In [2]:
# 7500  -> 100 minutes (1 hour 40 mins)
# 15000 -> 200 minutes (3 hours 20 mins)